### 1. Import necessary libraries

First, we'll import the libraries we need: `requests` for making HTTP requests, `BeautifulSoup` for parsing HTML, and `urljoin` from `urllib.parse` to handle relative URLs.

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os

### 2. Define the target URL

Let's define the URL of the webpage we want to scrape. For this example, I'll use a placeholder. **You should replace this with the actual URL of the website you want to scrape.**

In [2]:
# Replace this with the URL of the website you want to scrape
base_url = 'https://example.com/downloads'
print(f"Target URL: {base_url}")

Target URL: https://example.com/downloads


### 3. Fetch the webpage content

We'll use `requests.get()` to fetch the HTML content of the target URL. It's good practice to check if the request was successful (status code 200).

In [3]:
try:
    response = requests.get(base_url)
    response.raise_for_status() # Raise an exception for HTTP errors
    html_content = response.text
    print("Successfully fetched webpage content.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching webpage: {e}")
    html_content = None

Error fetching webpage: HTTPSConnectionPool(host='example.com', port=443): Max retries exceeded with url: /downloads (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))


### 4. Parse the HTML and find document links

Now, we'll use `BeautifulSoup` to parse the `html_content`. We'll look for `<a>` tags that contain `href` attributes pointing to document files like `.pdf` or `.txt`.

In [4]:
document_links = []
if html_content:
    soup = BeautifulSoup(html_content, 'html.parser')
    # Find all <a> tags
    for link in soup.find_all('a', href=True):
        href = link['href']
        # Check if the href points to a PDF or TXT file
        if href.endswith('.pdf') or href.endswith('.txt'):
            # Convert relative URL to an absolute URL
            absolute_url = urljoin(base_url, href)
            document_links.append(absolute_url)

if document_links:
    print(f"Found {len(document_links)} document links:")
    for doc_link in document_links:
        print(doc_link)
else:
    print("No document links (.pdf, .txt) found on the page.")
    print("Note: The example URL 'https://example.com/downloads' does not contain actual document links. You will need to replace `base_url` with a real target that has downloadable files for this to work.")

No document links (.pdf, .txt) found on the page.
Note: The example URL 'https://example.com/downloads' does not contain actual document links. You will need to replace `base_url` with a real target that has downloadable files for this to work.


### 5. Download the documents

Finally, we'll iterate through the found document links and download each file. We'll create a `downloads` directory to store them.

In [5]:
if document_links:
    # Create a directory to save downloads
    download_dir = 'downloads'
    os.makedirs(download_dir, exist_ok=True)
    print(f"Created directory: {download_dir}")

    for doc_url in document_links:
        try:
            print(f"Attempting to download: {doc_url}")
            doc_response = requests.get(doc_url, stream=True) # Use stream=True for large files
            doc_response.raise_for_status()

            # Extract filename from the URL
            filename = os.path.basename(doc_url)
            filepath = os.path.join(download_dir, filename)

            with open(filepath, 'wb') as f:
                for chunk in doc_response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Successfully downloaded: {filename} to {filepath}")

        except requests.exceptions.RequestException as e:
            print(f"Error downloading {doc_url}: {e}")
else:
    print("No documents to download.")

No documents to download.
